# Maia Lite Stage 2 — Restart-safe corpus + A100 SFT (v5)

**Dependency correction:** this version does not downgrade `huggingface-hub`, does not reinstall PyTorch, and does not modify unrelated Colab packages. It preserves verified corpus shards on Drive.

Run in order. CPU builds the corpus; A100 training begins only after the manifest is complete.

**Caution:** The original Stage 4 external serialization scripts were not supplied, so identical prompt serialization and linguistic retention have not been established. Review the training preflight before spending A100 credits.


In [ ]:
# 0 — Safe dependency preflight (NO forced upgrades/downgrades)
# Colab ships extra packages (Gradio/Diffusers) with potentially incompatible
# constraints. Do not downgrade huggingface-hub or reinstall torch here.
import sys, subprocess, importlib, importlib.metadata as md

required = {
    "transformers": "transformers>=4.46,<5",
    "datasets": "datasets>=3,<5",
    "peft": "peft>=0.14,<1",
    "accelerate": "accelerate>=1,<2",
    "huggingface_hub": "huggingface-hub>=0.27,<2",
    "safetensors": "safetensors>=0.4",
    "sympy": "sympy>=1.12",
    "tqdm": "tqdm",
}
missing=[]
for module, spec in required.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(spec)
if missing:
    print("Installing missing libraries only:", missing)
    subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
else:
    print("All required libraries are importable. No pip changes made.")

for module in ("torch", "transformers", "datasets", "peft", "accelerate", "huggingface_hub"):
    print(f"{module}: {md.version(module)}")

from packaging.version import Version
v=Version(md.version("huggingface_hub"))
if v < Version("0.27") or v >= Version("2.0"):
    raise RuntimeError("huggingface_hub is outside the supported range; stop before modifying packages.")
print("Dependency preflight complete. Existing unrelated pip conflicts are not modified.")

All required libraries are importable. No pip changes made.
torch: 2.11.0+cu130
transformers: 5.18.0
datasets: 4.8.5
peft: 0.21.1
accelerate: 1.15.0
huggingface_hub: 1.33.0
Dependency preflight complete. Existing unrelated pip conflicts are not modified.


In [ ]:
# 1 — Configuration, Drive, credentials, paths
import os, sys, json, re, csv, math, time, random, hashlib, shutil, tempfile, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from google.colab import drive
from tqdm.auto import tqdm

drive.mount('/content/drive')
PHASE = 'auto'  # 'auto', 'build', or 'train'
START_TRAINING = True  # Automatic on A100 only; no training during CPU corpus build
ROOT = Path('/content/drive/MyDrive/Colab_LLMs')
PROJECT = ROOT/'Maia_Lite_Stage2'
CORPUS = PROJECT/'sft_corpus_v3'
SHARDS = CORPUS/'sources'
SSD = Path('/content/maia_sft_v3')
STAGE4 = ROOT/'maia-sft-validation'/'data'/'stage4'
BASE_MODEL = PROJECT/'checkpoints'/'full'/'checkpoint-213624'
TOKENIZER_DIR = ROOT/'Maia_Lite_v3'/'tokenizer'
RUN_DRIVE = PROJECT/'sft_runs'/'stage2_multidomain_v3'
RUN_LOCAL = Path('/content/maia_sft_run_v3')
SEED = 42
random.seed(SEED)
# HF token from Colab Secrets (recommended), environment, or authenticated HF cache
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN') or None
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Hugging Face token available (value hidden).')
else:
    print('No HF_TOKEN found: public datasets can be downloaded anonymously; may be rate limited.')
for p in (CORPUS, SHARDS): p.mkdir(parents=True, exist_ok=True)
if PHASE == 'auto':
    PHASE = 'train' if (CORPUS/'manifest.json').exists() else 'build'
print('Selected phase:',PHASE)
if PHASE == 'train': RUN_DRIVE.mkdir(parents=True,exist_ok=True)
print('PHASE:',PHASE, 'Corpus:',CORPUS)

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path); temp=path.with_suffix(path.suffix+'.partial')
    with open(temp,'w',encoding='utf8') as f:
        json.dump(obj,f,ensure_ascii=False,indent=2);f.flush();os.fsync(f.fileno())
    os.replace(temp,path)

def stream_jsonl(path):
    with open(path,encoding='utf8') as f:
        for number,line in enumerate(f,1):
            if line.strip():
                try:yield json.loads(line)
                except Exception as e:raise ValueError(f'{path}:{number}: {e}')

def write_shard(name,producer):
    path=SHARDS/(name+'.jsonl'); meta=SHARDS/(name+'.meta.json')
    if path.exists() and meta.exists():
        try:
            m=json.loads(meta.read_text())
            if sha(path)==m['sha256'] and m['count']>0:
                print('REUSE',name,m['count']);return m
        except (ValueError, KeyError, OSError) as exc:
            print('REPAIR invalid metadata',name,str(exc))
    temp=path.with_suffix('.jsonl.partial'); count=0
    with open(temp,'w',encoding='utf8') as out:
        for obj in producer():
            out.write(json.dumps(obj,ensure_ascii=False)+'\n');count+=1
        out.flush();os.fsync(out.fileno())
    if count==0:raise RuntimeError(f'Shard {name} empty; source unavailable or incompatible')
    os.replace(temp,path)
    m={'name':name,'count':count,'sha256':sha(path),'file':path.name}
    atomic_json(meta,m)
    print('CREATED',name,count)
    return m

Mounted at /content/drive
Hugging Face token available (value hidden).
Selected phase: train
PHASE: train Corpus: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_corpus_v3


In [ ]:
# 2 — Source format normalization and mandatory frozen Stage 4 audit
# No guessing language from filenames; original records are preserved as messages where possible.
FROZEN_SHA = {
 'train.jsonl':'ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9',
 'validation.jsonl':'5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a',
 'test.jsonl':'03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834'
}

def normalize(row,domain,lang,source):
    if isinstance(row.get('messages'),list):
        msgs=[{'role':x['role'],'content':str(x['content'])} for x in row['messages']]
    elif 'instruction' in row and ('output' in row or 'response' in row):
        prompt=str(row['instruction'])+(('\n'+str(row['input'])) if row.get('input') else '')
        msgs=[{'role':'user','content':prompt},{'role':'assistant','content':str(row.get('output',row.get('response','')))}]
    elif ('prompt' in row or 'question' in row or 'inputs' in row) and ('response' in row or 'answer' in row or 'targets' in row or 'completion' in row):
        prompt=next(str(row[k]) for k in ('prompt','question','inputs') if k in row)
        response=next(str(row[k]) for k in ('response','answer','targets','completion') if k in row)
        msgs=[{'role':'user','content':prompt},{'role':'assistant','content':response}]
    else:raise ValueError(f'Unsupported source schema: {sorted(row.keys())}')
    if not msgs or msgs[-1]['role']!='assistant' or not msgs[-1]['content'].strip() or not any(x['role']=='user' for x in msgs):
        raise ValueError('Missing user or terminal assistant response')
    if any(x['role'] not in ('system','user','assistant') for x in msgs):raise ValueError('Unsupported role')
    return {'messages':msgs,'domain':domain,'language':lang,'source':source}

def original_stage4(split):
    path=STAGE4/(split+'.jsonl')
    if not path.exists():raise FileNotFoundError(f'Mandatory original Stage4 file missing: {path}')
    actual=sha(path)
    if actual!=FROZEN_SHA[path.name]:raise RuntimeError(f'Original Stage4 SHA mismatch {path}: {actual}')
    for row in stream_jsonl(path):
        # Language unknown in some Stage4 records; never silently invent a language label.
        lang=row.get('language',row.get('lang','und'))
        yield normalize(row,'linguistics',str(lang),'maia_stage4_'+split)

if PHASE=='build':
    for split in ('train','validation','test'):
        write_shard('stage4_'+split,lambda split=split:original_stage4(split))
    print('Stage4 sample:', next(stream_jsonl(SHARDS/'stage4_train.jsonl')))

In [ ]:
# 3 — Hugging Face datasets, streaming, pinned by dataset ID/config and captured revision
# Optional sources fail CLOSED: do not mark corpus complete if a requested dataset cannot be read.
from datasets import load_dataset
from huggingface_hub import HfApi
HF_SOURCES = [
 {'name':'gsm8k','id':'openai/gsm8k','config':'main','split':'train','domain':'mathematics','lang':'en','limit':7500},
 {'name':'python18k','id':'iamtarun/python_code_instructions_18k_alpaca','config':None,'split':'train','domain':'programming','lang':'en','limit':18000},
 {'name':'aya_pt','id':'CohereForAI/aya_dataset','config':None,'split':'train','domain':'linguistics','lang':'pt','limit':20000},
 {'name':'aya_en','id':'CohereForAI/aya_dataset','config':None,'split':'train','domain':'linguistics','lang':'en','limit':20000},
 {'name':'aya_es','id':'CohereForAI/aya_dataset','config':None,'split':'train','domain':'linguistics','lang':'es','limit':20000},
]
# Dataset license review is mandatory before public redistribution; source manifests retain IDs/revisions.
LANG_CODES={'pt':{'por','pt','Portuguese'},'en':{'eng','en','English'},'es':{'spa','es','Spanish'}}

def hf_rows(spec):
    api=HfApi(token=HF_TOKEN)
    info=api.dataset_info(spec['id'],token=HF_TOKEN)
    rev=info.sha
    print('SOURCE',spec['name'],'revision',rev,'license',info.cardData.get('license') if isinstance(info.cardData,dict) else 'inspect HF card')
    ds=load_dataset(spec['id'],name=spec['config'],split=spec['split'],streaming=True,revision=rev,token=HF_TOKEN)
    count=0
    for row in ds:
        if spec['name'].startswith('aya_'):
            if str(row.get('language_code') or row.get('language') or '').strip() not in LANG_CODES[spec['lang']]:continue
            if not isinstance(row.get('inputs'),str) or not isinstance(row.get('targets'),str) or not row['inputs'].strip() or not row['targets'].strip():continue
            obj=normalize({'prompt':str(row['inputs']).strip(),'response':str(row['targets']).strip()},'linguistics',spec['lang'],spec['id']+'@'+rev)
        elif spec['name']=='gsm8k':
            answer=str(row['answer']); parts=answer.rsplit('####',1)
            if len(parts)!=2:raise ValueError('GSM8K answer missing #### separator')
            explanation=parts[0].strip(); final=parts[1].strip()
            obj=normalize({'prompt':row['question'],'response':'Solution:\n'+explanation+'\n\nFinal answer: '+final},'mathematics','en',spec['id']+'@'+rev)
        else:
            obj=normalize(row,'programming','en',spec['id']+'@'+rev)
        yield obj
        count+=1
        if count>=spec['limit']:break

if PHASE=='build':
    for spec in HF_SOURCES:
        write_shard(spec['name'],lambda spec=spec:hf_rows(spec))

In [ ]:
# 4 — Deterministic verified synthetic math (120k) and Python (80k)
# Generators are deterministic across retries and require no API calls.
# Train/val/test use disjoint numeric seed streams; held-out templates should also be added later.
import ast

def math_rows(split,count):
    rng=random.Random({'train':31001,'validation':31002,'test':31003}[split])
    for i in range(count):
        op=i%6
        a=rng.randint(10,9999); b=rng.randint(2,999)
        if op==0:
            q=f'Calculate {a} + {b}. Show the calculation.'; ans=a+b
            sol=f'Add {a} and {b}. The sum is {ans}.'
        elif op==1:
            q=f'Calculate {a} - {b}. Show the calculation.';ans=a-b
            sol=f'Subtract {b} from {a}. The difference is {ans}.'
        elif op==2:
            a=rng.randint(2,999);b=rng.randint(2,99);ans=a*b
            q=f'Calculate {a} × {b}.';sol=f'Multiply {a} by {b}: {a} × {b} = {ans}.'
        elif op==3:
            ans=rng.randint(2,999);b=rng.randint(2,99);a=ans*b
            q=f'Calculate {a} ÷ {b}.';sol=f'Since {b} × {ans} = {a}, the quotient is {ans}.'
        elif op==4:
            x=rng.randint(-500,500); a=rng.randint(2,40);b=rng.randint(-100,100);c=a*x+b;ans=x
            q=f'Solve for x: {a}x + ({b}) = {c}.'
            sol=f'Subtract ({b}) from both sides: {a}x = {c-b}. Divide by {a}: x = {ans}.'
        else:
            pct=rng.choice([5,10,15,20,25,50]);base=rng.randrange(20,2001,20);ans=base*pct//100
            q=f'What is {pct}% of {base}?';sol=f'{pct}% of {base} = {base} × {pct}/100 = {ans}.'
        yield normalize({'prompt':q,'response':f'Solution:\n{sol}\n\nFinal answer: {ans}'},'mathematics','en','synthetic_math_v3_'+split)

def code_rows(split,count):
    rng=random.Random({'train':41001,'validation':41002,'test':41003}[split])
    for i in range(count):
        a=rng.randint(2,10000); b=rng.randint(2,999)
        k=i%8
        if k==0:
            q=f'Write a Python function that adds {a} to its argument.'
            src=f'def transform(x):\n    return x + {a}'
            expected=lambda x:x+a
        elif k==1:
            q=f'Write a Python function that multiplies its argument by {b}.'
            src=f'def transform(x):\n    return x * {b}'
            expected=lambda x:x*b
        elif k==2:
            q=f'Write a Python function that returns True if its integer argument is divisible by {b}.'
            src=f'def transform(x):\n    return x % {b} == 0'
            expected=lambda x:x%b==0
        elif k==3:
            q=f'Write a Python function that squares its numeric argument and adds {a}.'
            src=f'def transform(x):\n    return x*x + {a}'
            expected=lambda x:x*x+a
        elif k==4:
            q=f'Write a Python function that returns the absolute difference between its argument and {a}.'
            src=f'def transform(x):\n    return abs(x - {a})'
            expected=lambda x:abs(x-a)
        elif k==5:
            q=f'Write a Python function that returns the larger of its argument and {b}.'
            src=f'def transform(x):\n    return max(x, {b})'
            expected=lambda x:max(x,b)
        elif k==6:
            q=f'Write a Python function that returns the remainder when its integer argument is divided by {b}.'
            src=f'def transform(x):\n    return x % {b}'
            expected=lambda x:x%b
        else:
            q=f'Write a Python function that subtracts {b} from its argument.'
            src=f'def transform(x):\n    return x - {b}'
            expected=lambda x:x-b
        # Verify every generated program against an independently defined oracle.
        ast.parse(src)
        namespace={'__builtins__':{'abs':abs,'max':max}}
        exec(compile(src,'<generated>','exec'),namespace)
        for v in (-17,0,1,9,301):
            if namespace['transform'](v)!=expected(v):raise AssertionError('Generated code failed verification')
        yield normalize({'prompt':q,'response':'Python solution:\n```python\n'+src+'\n```'},'programming','en','synthetic_python_v3_'+split)

if PHASE=='build':
    for split,mc,pc in [('train',120000,80000),('validation',2500,2500),('test',2500,2500)]:
        write_shard('synthetic_math_'+split,lambda split=split,mc=mc:math_rows(split,mc))
        write_shard('synthetic_python_'+split,lambda split=split,pc=pc:code_rows(split,pc))

In [ ]:
# 5 — Assemble frozen train/validation/test and commit manifest LAST
# Exact dedup across splits; originals take priority. For stronger leakage checks use near-duplicate audits.
# No rebuild if committed corpus hashes validate.

def corpus_valid():
    mp=CORPUS/'manifest.json'
    if not mp.exists():return False
    try:
        m=json.loads(mp.read_text())
        return all((CORPUS/f).exists() and sha(CORPUS/f)==d['sha256'] for f,d in m['files'].items())
    except Exception:return False

if PHASE=='build':
    if corpus_valid():
        print('FROZEN CORPUS VALID — reuse; no regeneration.')
    else:
        mapping={
          'train':['stage4_train','gsm8k','python18k','aya_pt','aya_en','aya_es','synthetic_math_train','synthetic_python_train'],
          'validation':['stage4_validation','synthetic_math_validation','synthetic_python_validation'],
          'test':['stage4_test','synthetic_math_test','synthetic_python_test']
        }
        # Refuse to consolidate until every source and its metadata are valid.
        required=sorted({name for names in mapping.values() for name in names})
        problems=[]
        for name in required:
            p=SHARDS/(name+'.jsonl'); m=SHARDS/(name+'.meta.json')
            if not p.exists() or not m.exists(): problems.append(name+': missing file/metadata');continue
            try:
                info=json.loads(m.read_text())
                if sha(p)!=info['sha256'] or info['count']<1:problems.append(name+': bad hash/count')
            except Exception as e:problems.append(name+': '+str(e))
        if problems:raise RuntimeError('Cannot consolidate; repair only these shards: '+', '.join(problems))
        # canonicalize prompt+response for exact dedup; do not drop Stage4 duplicates silently
        seen=set(); stats={}; files={}; originals={}
        for split,names in mapping.items():
            tmp=CORPUS/(split+'.jsonl.partial'); target=CORPUS/(split+'.jsonl')
            counts=Counter(); kept_stage4=0; stage4_input=0
            with open(tmp,'w',encoding='utf8') as out:
                for name in names:
                    shard=SHARDS/(name+'.jsonl');meta=json.loads((SHARDS/(name+'.meta.json')).read_text())
                    if sha(shard)!=meta['sha256']:raise RuntimeError(f'Shard checksum mismatch: {name}')
                    for row in stream_jsonl(shard):
                        if name.startswith('stage4_'):stage4_input+=1
                        msgs=row['messages']; key=hashlib.sha256(json.dumps(msgs,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
                        if key in seen:
                            if name.startswith('stage4_'):raise RuntimeError('Original Stage4 duplicated across splits or within source; refusing to silently discard it')
                            continue
                        seen.add(key)
                        out.write(json.dumps(row,ensure_ascii=False)+'\n')
                        counts[(row['domain'],row['language'])]+=1
                        if name.startswith('stage4_'):kept_stage4+=1
                out.flush();os.fsync(out.fileno())
            if kept_stage4!=stage4_input:raise RuntimeError('Original linguistic data loss')
            os.replace(tmp,target)
            files[target.name]={'sha256':sha(target),'count':sum(counts.values())}
            stats[split]={'domains_languages':{str(k):v for k,v in counts.items()},'stage4_kept':kept_stage4}
            print(split,files[target.name],stats[split])
        manifest={'version':3,'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
                  'files':files,'stats':stats,'sources':[json.loads(p.read_text()) for p in sorted(SHARDS.glob('*.meta.json'))],
                  'notes':'Stage4 originals included without altering message contents. HF source IDs/revisions encoded per record. Review licensing before redistribution.'}
        atomic_json(CORPUS/'manifest.json',manifest)
        print('COMMITTED',CORPUS/'manifest.json')
assert corpus_valid(), 'Corpus incomplete: run build phase first; training is blocked.'

In [ ]:
# 6 — TRAIN PHASE: copy complete frozen corpus to local SSD and validate
if PHASE=='train':
    SSD.mkdir(parents=True,exist_ok=True)
    manifest=json.loads((CORPUS/'manifest.json').read_text())
    required_bytes=sum((CORPUS/f).stat().st_size for f in manifest['files'])
    free=shutil.disk_usage('/content').free
    if free<required_bytes*3:raise RuntimeError(f'Insufficient SSD space: need at least {required_bytes*3/1e9:.2f} GB')
    for f,meta in manifest['files'].items():
        src=CORPUS/f;dst=SSD/f
        if not dst.exists() or sha(dst)!=meta['sha256']:
            tmp=SSD/(f+'.partial');shutil.copyfile(src,tmp)
            if sha(tmp)!=meta['sha256']:raise RuntimeError('Copy corruption: '+f)
            os.replace(tmp,dst)
        print('SSD VERIFIED',f,meta['count'])
    shutil.copyfile(CORPUS/'manifest.json',SSD/'manifest.json')
    print('Ready to train using SSD only.')
else:print('Set PHASE="train" and rerun setup + this cell on your A100 session.')

SSD VERIFIED train.jsonl 178935
SSD VERIFIED validation.jsonl 3143
SSD VERIFIED test.jsonl 3147
Ready to train using SSD only.


In [ ]:
# ============================================================
# 7 — TOKENIZATION WITH PERSISTENT GOOGLE DRIVE CACHE
#
# MAIA LITE — STAGE 2 SFT
#
# Features:
#   - Original Maia tokenizer (50,257 tokens)
#   - Response-only loss masking
#   - Maximum context length: 1024
#   - Persistent tokenized cache on Google Drive
#   - SHA-256 integrity verification
#   - Automatic cache reuse after Colab restart
#   - Atomic writes
#
# IMPORTANT:
#   The serializer uses literal role labels.
#   Exact equivalence with Stage 4 is not yet proven.
# ============================================================

if PHASE == 'train':

    import os
    import json
    import hashlib
    import torch

    from pathlib import Path
    from collections import Counter
    from tqdm.auto import tqdm
    from transformers import AutoTokenizer, GPT2LMHeadModel
    from torch.utils.data import Dataset

    # --------------------------------------------------------
    # 1. VERIFY GPU
    # --------------------------------------------------------

    assert torch.cuda.is_available(), (
        'A100 GPU runtime required'
    )

    gpu_name = torch.cuda.get_device_name(0)

    print('GPU:', gpu_name)

    assert 'A100' in gpu_name, (
        'This configuration is designed for A100.'
    )

    # --------------------------------------------------------
    # 2. VERIFY MODEL AND TOKENIZER
    # --------------------------------------------------------

    for path in (BASE_MODEL, TOKENIZER_DIR):

        if not path.exists():
            raise FileNotFoundError(path)

    tok = AutoTokenizer.from_pretrained(
        str(TOKENIZER_DIR),
        use_fast=True
    )

    assert len(tok) == 50257, (
        f'Wrong tokenizer: {len(tok)}'
    )

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print('Tokenizer vocabulary:', len(tok))
    print('EOS:', tok.eos_token_id)
    print('PAD:', tok.pad_token_id)

    # --------------------------------------------------------
    # 3. TOKENIZATION CONFIGURATION
    # --------------------------------------------------------

    MAXLEN = 1024

    SERIALIZER_VERSION = 'maia_roles_v1'

    # --------------------------------------------------------
    # 4. SERIALIZE CONVERSATION
    # --------------------------------------------------------

    def serialize(row):

        parts = []

        for m in row['messages'][:-1]:

            parts.append(
                m['role'].capitalize()
                + ': '
                + m['content'].strip()
                + '\n'
            )

        prefix = ''.join(parts) + 'Assistant: '

        completion = (
            row['messages'][-1]['content'].strip()
            + tok.eos_token
        )

        return prefix, completion

    # --------------------------------------------------------
    # 5. TOKENIZE WITH RESPONSE-ONLY LABELS
    # --------------------------------------------------------

    def tokenize_row(row):

        prompt, answer = serialize(row)

        prompt_ids = tok(
            prompt,
            add_special_tokens=False
        )['input_ids']

        answer_ids = tok(
            answer,
            add_special_tokens=False
        )['input_ids']

        if not answer_ids:
            raise ValueError('Empty response')

        # Reject responses that cannot fit in context.
        if len(answer_ids) >= MAXLEN:
            return None

        # Preserve the complete response.
        # Truncate the beginning of the prompt if necessary.
        prompt_ids = prompt_ids[
            -(MAXLEN - len(answer_ids)):
        ]

        input_ids = prompt_ids + answer_ids

        labels = (
            [-100] * len(prompt_ids)
            + answer_ids
        )

        return {
            'input_ids': input_ids,
            'attention_mask': [1] * len(input_ids),
            'labels': labels,
            'domain': row['domain'],
            'language': row['language']
        }

    # --------------------------------------------------------
    # 6. PERSISTENT CACHE ON GOOGLE DRIVE
    # --------------------------------------------------------

    DRIVE_CACHE = Path(
        '/content/drive/MyDrive/'
        'Colab_LLMs/'
        'Maia_Lite_Stage2/'
        'sft_corpus_v3/'
        'tokenized'
    )

    DRIVE_CACHE.mkdir(
        parents=True,
        exist_ok=True
    )

    print('\nPersistent cache:')
    print(DRIVE_CACHE)

    # Use the persistent location for training files.
    cache = DRIVE_CACHE

    # --------------------------------------------------------
    # 7. TOKENIZE OR REUSE EXISTING CACHE
    # --------------------------------------------------------

    token_files = {}

    for split in ('train', 'validation', 'test'):

        out = cache / (split + '.jsonl')

        meta = cache / (split + '.meta.json')

        # Signature tied to source data and tokenization settings.
        signature_source = (
            manifest['files'][split + '.jsonl']['sha256']
            + str(MAXLEN)
            + str(len(tok))
            + SERIALIZER_VERSION
        )

        signature = hashlib.sha256(
            signature_source.encode()
        ).hexdigest()

        reuse = False

        if out.exists() and meta.exists():

            try:

                metadata = json.loads(
                    meta.read_text(encoding='utf-8')
                )

                reuse = (
                    metadata.get('signature') == signature
                    and sha(out) == metadata.get('sha256')
                )

            except (
                OSError,
                ValueError,
                KeyError
            ):

                reuse = False

        # ----------------------------------------------------
        # CACHE HIT
        # ----------------------------------------------------

        if reuse:

            print(
                f'REUSE tokenized {split}: '
                f'{metadata.get("count")} examples'
            )

        # ----------------------------------------------------
        # CACHE MISS
        # ----------------------------------------------------

        else:

            print(f'\nTokenizing {split}...')

            temp = cache / (split + '.partial')

            counts = Counter()

            n = 0

            with open(
                temp,
                'w',
                encoding='utf-8'
            ) as f:

                for row in tqdm(
                    stream_jsonl(
                        SSD / (split + '.jsonl')
                    ),
                    desc='Tokenizing ' + split,
                    total=manifest['files'][
                        split + '.jsonl'
                    ]['count']
                ):

                    item = tokenize_row(row)

                    if item is None:

                        counts['response_too_long'] += 1
                        continue

                    f.write(
                        json.dumps(
                            item,
                            ensure_ascii=False
                        ) + '\n'
                    )

                    n += 1

                f.flush()
                os.fsync(f.fileno())

            # Atomically publish the completed cache.
            os.replace(temp, out)

            # Record integrity metadata.
            atomic_json(
                meta,
                {
                    'signature': signature,
                    'sha256': sha(out),
                    'count': n,
                    'skipped': dict(counts)
                }
            )

            print(
                f'TOKENIZED {split}: {n} examples'
            )

            print(
                'Skipped:',
                dict(counts)
            )

        token_files[split] = out

    # --------------------------------------------------------
    # 8. FINAL INTEGRITY CHECK
    # --------------------------------------------------------

    print('\n' + '=' * 60)
    print('TOKENIZATION COMPLETED')
    print('=' * 60)

    for split, path in token_files.items():

        metadata = json.loads(
            (cache / (split + '.meta.json')).read_text(
                encoding='utf-8'
            )
        )

        print(
            f'{split:12s}: '
            f'{metadata["count"]:>8,} examples'
        )

    sample = next(
        stream_jsonl(token_files['train'])
    )

    print(
        '\nSample response-only mask:',
        sample['labels'][:24]
    )

    print('\nPersistent cache:', cache)

    print('\nREADY FOR CELL #8')

GPU: NVIDIA A100-SXM4-80GB
Tokenizer vocabulary: 50257
EOS: 2
PAD: 1

Persistent cache:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_corpus_v3/tokenized
REUSE tokenized train: 178532 examples
REUSE tokenized validation: 3134 examples
REUSE tokenized test: 3143 examples

TOKENIZATION COMPLETED
train       :  178,532 examples
validation  :    3,134 examples
test        :    3,143 examples

Sample response-only mask: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Persistent cache: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_corpus_v3/tokenized

READY FOR CELL #8


In [ ]:
# ============================================================
# MAIA LITE STAGE 2 — PRE-TRAINING SFT AUDIT
#
# Run AFTER cell #7 and BEFORE cell #8.
#
# Checks:
#   1. Token sequence length
#   2. Response-only masking
#   3. Prompt/response boundaries
#   4. EOS placement
#   5. Language and domain distribution
#   6. Sampling proportions
#   7. Representative decoded examples
#
# READ-ONLY: does not modify corpus or model.
# ============================================================

import json
import random
from collections import Counter, defaultdict

print("=" * 72)
print("MAIA LITE STAGE 2 — SFT INTEGRITY AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. Verify required variables
# ------------------------------------------------------------

assert "tok" in globals(), "Tokenizer not initialized"
assert "token_files" in globals(), "Tokenized files not found"
assert "MAXLEN" in globals(), "MAXLEN not defined"

print("\n[1] ENVIRONMENT")

print("Tokenizer vocabulary:", len(tok))
print("Maximum sequence length:", MAXLEN)
print("EOS token:", tok.eos_token_id)
print("PAD token:", tok.pad_token_id)

assert len(tok) == 50257
assert MAXLEN == 1024

# ------------------------------------------------------------
# 2. Load tokenized datasets
# ------------------------------------------------------------

print("\n[2] LOADING TOKENIZED DATASETS")

datasets = {}

for split in ("train", "validation", "test"):

    path = token_files[split]

    assert path.exists(), f"Missing: {path}"

    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    datasets[split] = rows

    print(f"{split:12s}: {len(rows):,} examples")

# ------------------------------------------------------------
# 3. Verify token lengths and masks
# ------------------------------------------------------------

print("\n[3] TOKENIZATION AND MASK AUDIT")

errors = Counter()
statistics = defaultdict(list)

for split, rows in datasets.items():

    for index, row in enumerate(rows):

        ids = row["input_ids"]
        labels = row["labels"]
        attention = row["attention_mask"]

        n = len(ids)

        if not (
            len(ids) == len(labels) == len(attention)
        ):
            errors["length_mismatch"] += 1
            continue

        if n > MAXLEN:
            errors["sequence_too_long"] += 1

        if n == 0:
            errors["empty_sequence"] += 1
            continue

        if any(x != 1 for x in attention):
            errors["invalid_attention_mask"] += 1

        active = [
            i for i, label in enumerate(labels)
            if label != -100
        ]

        if not active:
            errors["fully_masked_response"] += 1
            continue

        first = active[0]

        if first == 0:
            errors["unmasked_prompt_or_empty_prefix"] += 1

        if any(x != -100 for x in labels[:first]):
            errors["invalid_prompt_mask"] += 1

        if any(x == -100 for x in labels[first:]):
            errors["interrupted_response_mask"] += 1

        if labels[first:] != ids[first:]:
            errors["response_label_mismatch"] += 1

        if ids[-1] != tok.eos_token_id:
            errors["missing_eos"] += 1

        statistics[split].append({
            "total": n,
            "prompt": first,
            "response": len(active),
        })

for split, values in statistics.items():

    if not values:
        continue

    total = sum(x["total"] for x in values)
    prompt = sum(x["prompt"] for x in values)
    response = sum(x["response"] for x in values)

    print(f"\n{split.upper()}")

    print("Valid examples:", len(values))
    print("Mean sequence length:", round(total / len(values), 1))
    print("Mean prompt tokens:", round(prompt / len(values), 1))
    print("Mean response tokens:", round(response / len(values), 1))
    print("Response token fraction:", round(100 * response / total, 2), "%")
    print("Maximum sequence length:", max(x["total"] for x in values))

# ------------------------------------------------------------
# 4. Language and domain distribution
# ------------------------------------------------------------

print("\n[4] TRAINING DISTRIBUTION")

train = datasets["train"]

domain_counts = Counter(
    row["domain"] for row in train
)

language_counts = Counter(
    row["language"] for row in train
)

combined_counts = Counter(
    (row["domain"], row["language"])
    for row in train
)

print("\nDomains:")

for domain, count in sorted(domain_counts.items()):

    print(
        f"  {domain:15s}: "
        f"{count:7,d} "
        f"({100 * count / len(train):5.1f}%)"
    )

print("\nLanguages:")

for lang, count in sorted(language_counts.items()):

    print(
        f"  {lang:5s}: "
        f"{count:7,d} "
        f"({100 * count / len(train):5.1f}%)"
    )

print("\nDomain / language:")

for (domain, lang), count in sorted(combined_counts.items()):

    print(
        f"  {domain:15s} {lang:5s}: {count:7,d}"
    )

# ------------------------------------------------------------
# 5. Verify balanced sampling configuration
# ------------------------------------------------------------

print("\n[5] BALANCED SAMPLING PROJECTION")

fractions = {
    "linguistics": 0.40,
    "mathematics": 0.30,
    "programming": 0.30,
}

target = max(len(train), 300000)

assert abs(sum(fractions.values()) - 1.0) < 1e-9

for domain, weight in fractions.items():

    available = domain_counts.get(domain, 0)

    assert available > 0, f"Empty domain: {domain}"

    expected = round(target * weight)

    print(
        f"{domain:15s}: "
        f"{expected:7,d} sampled examples "
        f"({weight * 100:.0f}%) "
        f"from {available:,} available"
    )

print("\nTotal planned samples per epoch:", target)

print(
    "NOTE: This verifies the intended proportions, "
    "not the actual Trainer execution."
)

# ------------------------------------------------------------
# 6. Decode representative examples
# ------------------------------------------------------------

print("\n[6] DECODED TRAINING EXAMPLES")

random.seed(42)

groups = defaultdict(list)

for row in train:

    key = (row["domain"], row["language"])

    groups[key].append(row)

for key in sorted(groups):

    domain, language = key

    row = random.choice(groups[key])

    ids = row["input_ids"]
    labels = row["labels"]

    first = next(
        i for i, value in enumerate(labels)
        if value != -100
    )

    prompt_text = tok.decode(
        ids[:first],
        skip_special_tokens=False
    )

    response_text = tok.decode(
        ids[first:],
        skip_special_tokens=False
    )

    print("\n" + "-" * 72)

    print("DOMAIN:", domain)
    print("LANGUAGE:", language)

    print("\nPROMPT:")
    print(prompt_text[:500])

    print("\nRESPONSE:")
    print(response_text[:700])

    print("\nTOKENS:")
    print("Prompt:", first)
    print("Response:", len(ids) - first)
    print("Total:", len(ids))

# ------------------------------------------------------------
# 7. Final report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL INTEGRITY REPORT")
print("=" * 72)

if errors:

    print("\nERRORS DETECTED:")

    for name, count in errors.items():
        print(f"  {name}: {count:,}")

    print("\nSTATUS: FAILED")
    print("DO NOT START TRAINING.")

else:

    print("\nAll structural tokenization checks passed.")

    print("No sequences exceed the model context window.")
    print("No fully masked responses.")
    print("Response labels match response token IDs.")
    print("All checked sequences terminate with EOS.")

    print("\nSTATUS: STRUCTURAL AUDIT PASSED")

    print(
        "\nIMPORTANT: The original Stage 4 conversation "
        "serialization has not yet been verified."
    )

print("\nTRAINING WAS NOT STARTED.")
print("=" * 72)

MAIA LITE STAGE 2 — SFT INTEGRITY AUDIT

[1] ENVIRONMENT
Tokenizer vocabulary: 50257
Maximum sequence length: 1024
EOS token: 2
PAD token: 1

[2] LOADING TOKENIZED DATASETS
train       : 178,532 examples
validation  : 3,134 examples
test        : 3,143 examples

[3] TOKENIZATION AND MASK AUDIT

TRAIN
Valid examples: 178532
Mean sequence length: 102.1
Mean prompt tokens: 45.2
Mean response tokens: 56.9
Response token fraction: 55.72 %
Maximum sequence length: 1024

VALIDATION
Valid examples: 3134
Mean sequence length: 160.3
Mean prompt tokens: 90.7
Mean response tokens: 69.6
Response token fraction: 43.4 %
Maximum sequence length: 1024

TEST
Valid examples: 3143
Mean sequence length: 153.9
Mean prompt tokens: 85.7
Mean response tokens: 68.2
Response token fraction: 44.31 %
Maximum sequence length: 1024

[4] TRAINING DISTRIBUTION

Domains:
  linguistics    :  32,711 ( 18.3%)
  mathematics    : 104,083 ( 58.3%)
  programming    :  41,738 ( 23.4%)

Languages:
  en   : 156,124 ( 87.4%)
  es

In [ ]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# ============================================================
# 8 — MAIA LITE STAGE 2: LoRA SFT TRAINING
#
# Balanced sampling:
#   Linguistics: 40%
#   Mathematics: 30%
#   Programming: 30%
#
# Base model: Maia Lite checkpoint-213624
# LoRA: r=16, alpha=32, dropout=0.05
# Effective batch: 8
# Epochs: 2
# Learning rate: 2e-5
# Warmup: 3%
#
# Full Trainer checkpoints copied to Google Drive.
# Original pretrained checkpoint is never modified.
# ============================================================

if PHASE == 'train':

    import os
    import json
    import math
    import random
    import shutil
    import inspect
    import torch

    from collections import defaultdict
    from torch.utils.data import Dataset
    from transformers import (
        Trainer,
        TrainingArguments,
        TrainerCallback,
        GPT2LMHeadModel
    )
    from peft import (
        LoraConfig,
        get_peft_model,
        TaskType
    )

    # --------------------------------------------------------
    # 1. LOAD TOKENIZED DATA
    # --------------------------------------------------------

    class Encoded(Dataset):

        def __init__(self, path):
            self.rows = list(stream_jsonl(path))

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, i):
            return self.rows[i]

    full = Encoded(token_files['train'])

    by_domain = defaultdict(list)

    for row in full.rows:
        by_domain[row['domain']].append(row)

    print(
        'TRAIN DOMAINS',
        {k: len(v) for k, v in by_domain.items()}
    )

    assert all(
        by_domain[k]
        for k in (
            'linguistics',
            'mathematics',
            'programming'
        )
    )

    # --------------------------------------------------------
    # 2. DETERMINISTIC BALANCED SAMPLING
    # --------------------------------------------------------

    target = max(len(full), 300000)

    rng = random.Random(SEED)

    fractions = {
        'linguistics': 0.40,
        'mathematics': 0.30,
        'programming': 0.30
    }

    sampled = []

    for domain, weight in fractions.items():

        pool = by_domain[domain]

        sampled.extend(
            rng.choices(
                pool,
                k=round(target * weight)
            )
        )

    rng.shuffle(sampled)

    print('Samples per epoch:', len(sampled))

    class Samples(Dataset):

        def __len__(self):
            return len(sampled)

        def __getitem__(self, i):

            x = sampled[i]

            return {
                k: x[k]
                for k in (
                    'input_ids',
                    'attention_mask',
                    'labels'
                )
            }

    # --------------------------------------------------------
    # 3. DYNAMIC PADDING
    # --------------------------------------------------------

    def collate(batch):

        L = max(
            len(x['input_ids'])
            for x in batch
        )

        result = {}

        for key in (
            'input_ids',
            'attention_mask',
            'labels'
        ):

            if key == 'input_ids':
                pad_value = tok.pad_token_id

            elif key == 'attention_mask':
                pad_value = 0

            else:
                pad_value = -100

            result[key] = torch.tensor(
                [
                    x[key]
                    + [pad_value] * (L - len(x[key]))
                    for x in batch
                ],
                dtype=torch.long
            )

        return result

    # --------------------------------------------------------
    # 4. LOAD MAIA LITE BASE MODEL
    # --------------------------------------------------------

    print('\nLoading Maia Lite base model...')

    model = GPT2LMHeadModel.from_pretrained(
        str(BASE_MODEL),
        torch_dtype=torch.float32
    )

    # --------------------------------------------------------
    # 5. APPLY LoRA
    # --------------------------------------------------------

    lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            'c_attn',
            'c_proj',
            'c_fc'
        ]
    )

    model = get_peft_model(
        model,
        lora
    )

    model.config.use_cache = False

    model.gradient_checkpointing_enable()

    model.print_trainable_parameters()

    # --------------------------------------------------------
    # 6. CHECKPOINT SYNCHRONIZATION
    # --------------------------------------------------------

    class SyncCheckpoint(TrainerCallback):

        def on_save(
            self,
            args,
            state,
            control,
            **kwargs
        ):

            src = (
                RUN_LOCAL
                / f'checkpoint-{state.global_step}'
            )

            dst = RUN_DRIVE / src.name

            if not src.exists():
                return control

            temp = RUN_DRIVE / (
                src.name + '.partial'
            )

            if temp.exists():
                shutil.rmtree(temp)

            shutil.copytree(
                src,
                temp
            )

            if dst.exists():
                shutil.rmtree(dst)

            os.replace(
                temp,
                dst
            )

            atomic_json(
                RUN_DRIVE / 'latest.json',
                {
                    'step': state.global_step,
                    'checkpoint': src.name
                }
            )

            print(
                'DRIVE CHECKPOINT COMMITTED:',
                dst
            )

            return control

    # --------------------------------------------------------
    # 7. CHECK FOR RESUMABLE CHECKPOINT
    # --------------------------------------------------------

    RUN_LOCAL.mkdir(
        parents=True,
        exist_ok=True
    )

    RUN_DRIVE.mkdir(
        parents=True,
        exist_ok=True
    )

    last = RUN_DRIVE / 'latest.json'

    resume = None

    if last.exists():

        info = json.loads(
            last.read_text()
        )

        src = (
            RUN_DRIVE
            / info['checkpoint']
        )

        dst = (
            RUN_LOCAL
            / info['checkpoint']
        )

        if not src.is_dir():

            raise RuntimeError(
                'Latest checkpoint missing: '
                + str(src)
            )

        if not dst.exists():

            shutil.copytree(
                src,
                dst
            )

        names = {
            p.name
            for p in dst.iterdir()
        }

        required = {
            'trainer_state.json',
            'optimizer.pt',
            'scheduler.pt'
        }

        if not required.issubset(names):

            raise RuntimeError(
                'Incomplete checkpoint: '
                + str(required - names)
            )

        if not any(
            x.startswith('rng_state')
            for x in names
        ):

            raise RuntimeError(
                'RNG state missing; '
                'refusing unsafe resume'
            )

        resume = str(dst)

        print(
            'RESUME FROM:',
            resume
        )

    # --------------------------------------------------------
    # 8. TRAINING CONFIGURATION
    #
    # Compatibility fix:
    #   warmup_ratio -> warmup_steps
    #
    # No package installation or runtime restart.
    # --------------------------------------------------------

    micro_batch = 1

    accumulation = 8

    epochs = 2

    steps_per_epoch = math.ceil(
        len(sampled)
        / (micro_batch * accumulation)
    )

    total_steps = (
        steps_per_epoch * epochs
    )

    warmup_steps = math.ceil(
        total_steps * 0.03
    )

    print('\nTRAINING CONFIGURATION')

    print(
        'Optimizer steps:',
        total_steps
    )

    print(
        'Warmup steps:',
        warmup_steps
    )

    config = dict(

        output_dir=str(RUN_LOCAL),

        per_device_train_batch_size=1,

        gradient_accumulation_steps=8,

        num_train_epochs=2,

        learning_rate=2e-5,

        weight_decay=0.01,

        warmup_steps=warmup_steps,

        lr_scheduler_type='cosine',

        bf16=True,

        fp16=False,

        logging_steps=100,

        save_steps=1000,

        save_strategy='steps',

        save_total_limit=3,

        report_to='none',

        remove_unused_columns=False,

        seed=SEED,

        gradient_checkpointing=True,

        dataloader_num_workers=2,

        optim='adamw_torch'
    )

    # --------------------------------------------------------
    # 9. CHECK INSTALLED TRANSFORMERS API
    # --------------------------------------------------------

    accepted = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    unsupported = sorted(
        set(config) - set(accepted)
    )

    if unsupported:

        raise RuntimeError(
            'TrainingArguments incompatible '
            'with this environment: '
            + ', '.join(unsupported)
            + '. No training started.'
        )

    args = TrainingArguments(
        **config
    )

    # --------------------------------------------------------
    # 10. CREATE TRAINER
    # --------------------------------------------------------

    trainer = Trainer(

        model=model,

        args=args,

        train_dataset=Samples(),

        data_collator=collate,

        callbacks=[
            SyncCheckpoint()
        ]
    )

    print('\n' + '=' * 60)

    print('MAIA LITE STAGE 2 — READY')

    print('=' * 60)

    print(
        'Samples per epoch:',
        len(sampled)
    )

    print(
        'Resume checkpoint:',
        resume
    )

    # --------------------------------------------------------
    # 11. START TRAINING
    # --------------------------------------------------------

    if START_TRAINING:

        print('\nSTARTING TRAINING...\n')

        trainer.train(
            resume_from_checkpoint=resume
        )

    else:

        print(
            '\nTRAINING NOT STARTED.'
        )

        print(
            'Set START_TRAINING=True '
            'in cell 1 to begin.'
        )

TRAIN DOMAINS {'linguistics': 32711, 'mathematics': 104083, 'programming': 41738}
Samples per epoch: 300000

Loading Maia Lite base model...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 6,291,456 || all params: 361,114,624 || trainable%: 1.7422
RESUME FROM: /content/maia_sft_run_v3/checkpoint-38000

TRAINING CONFIGURATION
Optimizer steps: 75000
Warmup steps: 2250

MAIA LITE STAGE 2 — READY
Samples per epoch: 300000
Resume checkpoint: /content/maia_sft_run_v3/checkpoint-38000

STARTING TRAINING...



[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
38100,1.360233
38200,1.388517
38300,1.481439
38400,1.389234
38500,1.344561
38600,1.402150
38700,1.443247
38800,1.425255
38900,1.438716
39000,1.462813


DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-39000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-40000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-41000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-42000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-43000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-44000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-45000
DRIVE CHECKPOINT COMMITTED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/c

In [ ]:
# 9 — Evaluation, quality gates, and runbook
# The following cell evaluates held-out token losses; writing quality requires human review.
if PHASE=='train':
    import torch
    from torch.utils.data import DataLoader
    @torch.no_grad()
    def loss_by_domain(model,path,limit_per_domain=150):
        model.eval();sums=defaultdict(float);counts=Counter()
        for row in stream_jsonl(path):
            key=(row['domain'],row['language'])
            if counts[key]>=limit_per_domain:continue
            batch=collate([{k:row[k] for k in ('input_ids','attention_mask','labels')}])
            batch={k:v.to(model.device) for k,v in batch.items()}
            loss=model(**batch).loss.item()
            if math.isfinite(loss):sums[key]+=loss;counts[key]+=1
        model.train()
        return {str(k):{'loss':sums[k]/counts[k],'n':counts[k]} for k in counts if counts[k]}
    print('Held-out validation losses:',json.dumps(loss_by_domain(model,token_files['validation']),indent=2))
    print('IMPORTANT: Original Stage4 language labels may be und; independently evaluate PT/EN/ES prompts and long-form writing.')

## Restart instructions

1. Run all cells on CPU for the initial build. The notebook reuses verified `sources/*.jsonl` + `.meta.json` files (including your 12 existing valid shards). It now extracts Aya explicitly from `inputs` and `targets`, avoiding the `messages` schema collision. The only missing sources should be `aya_en` and `aya_es`.
2. A valid `manifest.json` means the corpus is complete. On a new runtime, `PHASE='auto'` chooses training automatically. Switch to A100, run all cells, and the corpus will be copied from Drive to local SSD, with SHA-256 verification.
3. **Training starts automatically** on A100 after copying and verifying the frozen corpus. Set `START_TRAINING=False` only if you want a dry run. Training checkpoints are synchronized to Drive. Resume uses optimizer/scheduler/RNG state when present.
4. The tokenizer comes from Maia Lite v3 (`50257` tokens). The pretraining checkpoint remains unchanged.
5. **Scientific limitation:** the prior successful Stage 4 SFT serializer lives in external source files; the serializer used here is not proven equivalent. Inspect language generations in PT/EN/ES and compare against the prior v3 SFT before accepting the adapter. Dataset licensing must be reviewed before redistribution.
